In [ ]:
import os
import json
import glob
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict, Counter

DATA_DIR = Path('../data/processed/waymo_e2e/val')
LABEL_JSON = Path('../data/val_sequence_name_to_scenario_cluster.json')

print(f'Data dir exists: {DATA_DIR.exists()}')
print(f'Label JSON exists: {LABEL_JSON.exists()}')

# Examine the contents of a single npz file

In [ ]:
npz_files = sorted(DATA_DIR.glob('**/*.npz'))
file = npz_files[0]

In [ ]:
file.name

In [ ]:
data = np.load(file, allow_pickle=True)
data

In [ ]:
data.files

In [ ]:
for key in data.files:
    print(key)
    print(data[key])
    print('\n')

## Take a look at _modality_data
- Past states and future states have position and velocity info that may be useful in the future
- Intent: go straight, go right, go left... will ignore for now

In [ ]:
list(data['_modality_data'].item().keys())

In [ ]:
for key in list(data['_modality_data'].item().keys()):
    print(key)
    print(data['_modality_data'].item()[key])
    print('\n')

In [ ]:
def load_image(fpath):
    data = np.load(fpath, allow_pickle=True)
    modality = data['_modality_data'].item()
    keys = list(modality.keys())
    img = modality[keys[0]]  # CAMERAS is always first key
    if img.dtype in [np.float32, np.float64]:
        img = np.clip(img, 0, 1)
    else:
        img = img.astype(np.uint8)
    return img

    
# Test it
img = load_image(npz_files[0])
print(f'Image shape: {img.shape}, dtype: {img.dtype}')
plt.figure(figsize=(14, 3))
plt.imshow(img)
plt.axis('off')
plt.title(f'Sample frame — {npz_files[0].name}')
plt.tight_layout()
plt.show()

## Find Class Label data

In [ ]:
LABEL_JSON

In [ ]:
with open(LABEL_JSON) as f:
    labels = json.load(f)

In [ ]:
# Peek at the JSON structure
first_key = list(labels.keys())[0]
print(f'Key: {first_key}')
print(f'Value: {labels[first_key]}')

In [ ]:
label_name, label_frame = npz_files[0].name.split('_')
label_frame = label_frame[:-4]
print(label_name)
print(label_frame)

In [ ]:
labels[npz_files[0].name[:-8]]

## Find unique 'videos'
- split npz file name on '_'
- first item is label name
- second item is the frame name

In [ ]:
# Dictionary to store each unique video/sequence and its frame numbers
seq_to_frames = {}
bad_files = []

print("total npz files:", len(npz_files))

# Find unique videos/sequences
for filepath in npz_files:
    
    # Get just the file name without the .npz extension
    filename = filepath.stem

    # Split on the underscore
    sequence_id, frame_id_text = filename.rsplit("_", 1)

    # Make sure the frame id is actually numeric
    # This skips files like: 0075..._91(1).npz
    if not frame_id_text.isdigit():
        bad_files.append(filepath)
        continue
    
    # Convert frame_id from string to integer
    frame_id = int(frame_id_text)

    # If this sequence_id has not been seen yet, create an empty list for it
    if sequence_id not in seq_to_frames:
        seq_to_frames[sequence_id] = []

    # Add the frame number to that sequence's frame list
    seq_to_frames[sequence_id].append(frame_id)

# Sort the frame IDs for each sequence
for sequence_id in seq_to_frames:
    seq_to_frames[sequence_id] = sorted(seq_to_frames[sequence_id])

print("unique videos/sequences:", len(seq_to_frames))
print("bad/skipped files:", len(bad_files))

In [ ]:
seq_to_frames

## For fun, make a gif
- frame rate is 4Hz

In [ ]:
import numpy as np
import imageio.v2 as imageio
from pathlib import Path

# Relative path from your notebook folder to the gifs folder
gif_dir = Path("../data/gifs")

# Create the folder if it does not exist
gif_dir.mkdir(parents=True, exist_ok=True)

print("GIFs will be saved to:")
print(gif_dir.resolve())

In [ ]:
# Build this once before the GIF loop
frame_lookup = {}
bad_files = []

for filepath in npz_files:
    filename = filepath.stem

    try:
        seq_id, frame_id_text = filename.rsplit("_", 1)
    except ValueError:
        bad_files.append(filepath)
        continue

    # Skip duplicate/malformed files like ..._91(1).npz
    if not frame_id_text.isdigit():
        bad_files.append(filepath)
        continue

    frame_id = int(frame_id_text)
    frame_lookup[(seq_id, frame_id)] = filepath

print("valid files:", len(frame_lookup))
print("bad files skipped:", len(bad_files))

In [ ]:
gif_count = 1
missing_count = 0

for sequence_id, frame_ids in seq_to_frames.items():

    print(f"Creating gif {gif_count} of {len(seq_to_frames)}")

    frames_for_gif = []

    for frame_id in sorted(frame_ids):

        key = (sequence_id, frame_id)

        if key not in frame_lookup:
            print(f"  Missing frame {frame_id} for sequence {sequence_id}")
            missing_count += 1
            continue

        fpath = frame_lookup[key]
        img = load_image(fpath)

        # Convert image to uint8 for GIF writing
        if img.dtype != np.uint8:
            if img.max() <= 1.0:
                img = (img * 255).astype(np.uint8)
            else:
                img = img.astype(np.uint8)

        frames_for_gif.append(img)

    if len(frames_for_gif) == 0:
        print(f"  No frames found for {sequence_id}, skipping.")
        gif_count += 1
        continue

    # Save using relative path
    gif_path = gif_dir / f"{sequence_id}.gif"

    imageio.mimsave(gif_path, frames_for_gif, duration=0.15)

    print(f"  Saved to: {gif_path}")

    gif_count += 1

print("Done.")
print("Missing frames:", missing_count)
print("Bad files skipped:", len(bad_files))